
# Deep Learning Re-analysis of the Heart Failure Dataset

## Objective

This notebook evaluates whether a **small neural network (MLP)** can improve prediction of `DEATH_EVENT` on the 299-patient heart-failure dataset.

The notebook is deliberately separate from the classical ML re-analysis.

### Important methodological rules

- `time` is **excluded** from classification features because it represents follow-up/survival time and can leak outcome-related information into a simple death-event classifier.
- Preprocessing is fitted **inside the training data/fold only**.
- Categorical/binary variables are handled separately from continuous variables.
- Class imbalance is handled using **class-weighted loss** rather than generating synthetic patients for the neural network.
- The primary evaluation uses **repeated stratified cross-validation**, not a single lucky train/test split.
- The MLP is intentionally small because the dataset contains only 299 patients.


In [5]:

# 1. Imports and reproducibility

import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, average_precision_score,
    log_loss, brier_score_loss, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)


PyTorch: 2.11.0+cpu
Device: cpu



## 2. Load and validate the dataset

Expected file:

`heart_failure_clinical_records_dataset.csv`

The dataset should contain 299 patients and the target `DEATH_EVENT`.


In [9]:

# 2. Load dataset
import pandas as pd

import os
candidates = ["heart_failure_clinical_records_dataset.csv", "../data/heart_failure_clinical_records_dataset.csv", "../../data/heart_failure_clinical_records_dataset.csv"]
DATA_PATH = next((p for p in candidates if os.path.exists(p)), "heart_failure_clinical_records_dataset.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())

required_columns = {
    "age", "anaemia", "creatinine_phosphokinase", "diabetes",
    "ejection_fraction", "high_blood_pressure", "platelets",
    "serum_creatinine", "serum_sodium", "sex", "smoking",
    "time", "DEATH_EVENT"
}

missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

print("\nTarget distribution:")
print(df["DEATH_EVENT"].value_counts())
print("\nMissing values:")
print(df.isna().sum())


FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/VIVEK/Desktop/heart_failure_clinical_records_dataset.csv'


## 3. Define classification features

We exclude `time`.

### Input variables

- age
- anaemia
- creatinine_phosphokinase
- diabetes
- ejection_fraction
- high_blood_pressure
- platelets
- serum_creatinine
- serum_sodium
- sex
- smoking

### Target

`DEATH_EVENT`

This is important because `time` is a follow-up/survival variable rather than an independent baseline clinical feature for this classification task.


In [ ]:

FEATURES = [
    "age",
    "anaemia",
    "creatinine_phosphokinase",
    "diabetes",
    "ejection_fraction",
    "high_blood_pressure",
    "platelets",
    "serum_creatinine",
    "serum_sodium",
    "sex",
    "smoking"
]

TARGET = "DEATH_EVENT"

X = df[FEATURES].copy()
y = df[TARGET].astype(int).copy()

BINARY_FEATURES = [
    "anaemia", "diabetes", "high_blood_pressure", "sex", "smoking"
]

CONTINUOUS_FEATURES = [
    "age",
    "creatinine_phosphokinase",
    "ejection_fraction",
    "platelets",
    "serum_creatinine",
    "serum_sodium"
]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures:", FEATURES)



## 4. Small MLP architecture

Because the dataset is small, the network is intentionally compact:

```text
11 clinical inputs
       ↓
Dense(32) + ReLU
       ↓
Dropout(0.20)
       ↓
Dense(16) + ReLU
       ↓
Dropout(0.20)
       ↓
Dense(1)
```

We use `BCEWithLogitsLoss` with a positive-class weight calculated from the training fold.


In [ ]:

class HeartFailureMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


def make_model(input_dim):
    return HeartFailureMLP(input_dim).to(DEVICE)



## 5. Training function

The scaler is fitted only on the training portion.

The validation/test portion is transformed using the training scaler.


In [ ]:

def train_mlp(
    X_train,
    y_train,
    X_valid,
    y_valid,
    epochs=250,
    batch_size=32,
    learning_rate=1e-3,
    weight_decay=1e-4,
    patience=30,
    seed=42
):
    seed_everything(seed)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_valid_scaled = scaler.transform(X_valid)

    X_train_tensor = torch.tensor(
        X_train_scaled, dtype=torch.float32
    ).to(DEVICE)

    y_train_tensor = torch.tensor(
        y_train.values, dtype=torch.float32
    ).to(DEVICE)

    X_valid_tensor = torch.tensor(
        X_valid_scaled, dtype=torch.float32
    ).to(DEVICE)

    y_valid_tensor = torch.tensor(
        y_valid.values, dtype=torch.float32
    ).to(DEVICE)

    train_ds = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(
        train_ds,
        batch_size=min(batch_size, len(train_ds)),
        shuffle=True
    )

    model = make_model(X_train.shape[1])

    n_positive = int(y_train.sum())
    n_negative = int(len(y_train) - n_positive)

    pos_weight = torch.tensor(
        [n_negative / max(n_positive, 1)],
        dtype=torch.float32,
        device=DEVICE
    )

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    best_state = None
    best_valid_loss = np.inf
    epochs_without_improvement = 0

    history = {
        "train_loss": [],
        "valid_loss": []
    }

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        model.eval()

        with torch.no_grad():
            valid_logits = model(X_valid_tensor)
            valid_loss = criterion(valid_logits, y_valid_tensor).item()

        train_loss = float(np.mean(train_losses))

        history["train_loss"].append(train_loss)
        history["valid_loss"].append(valid_loss)

        if valid_loss < best_valid_loss - 1e-5:
            best_valid_loss = valid_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, scaler, history



## 6. Metrics

We evaluate more than accuracy.

For an imbalanced clinical dataset, ROC-AUC, PR-AUC, F1, balanced accuracy, log loss and Brier score are important.


In [ ]:

def evaluate_predictions(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(y_true, pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, prob),
        "PR_AUC": average_precision_score(y_true, prob),
        "LogLoss": log_loss(y_true, np.clip(prob, 1e-7, 1-1e-7)),
        "Brier": brier_score_loss(y_true, prob)
    }



# 7. Single stratified 70/30 experiment

This is included for comparison with the paper's 70/30 design.

It should **not** be treated as the final estimate because a single split can be unstable with only 299 patients.


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

mlp, scaler, history = train_mlp(
    X_train,
    y_train,
    X_test,
    y_test,
    epochs=300,
    batch_size=32,
    learning_rate=1e-3,
    weight_decay=1e-4,
    patience=40,
    seed=42
)

mlp.eval()

with torch.no_grad():
    X_test_scaled = scaler.transform(X_test)
    X_test_tensor = torch.tensor(
        X_test_scaled, dtype=torch.float32
    ).to(DEVICE)

    logits = mlp(X_test_tensor)
    test_prob = torch.sigmoid(logits).cpu().numpy()

single_result = evaluate_predictions(y_test, test_prob)

pd.DataFrame([single_result], index=["MLP_70_30"])


In [ ]:

# Training curves

plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Training loss")
plt.plot(history["valid_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("MLP Training Curve")
plt.legend()
plt.show()



# 8. Repeated Stratified Cross-Validation

This is the **primary evaluation**.

We use:

- 10 folds
- 5 repeats
- 50 validation evaluations

Each fold independently fits its scaler and neural network.

This reduces dependence on one arbitrary train/test split.


In [ ]:

def repeated_cv_mlp(
    X,
    y,
    n_splits=10,
    n_repeats=5,
    epochs=250
):
    cv = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=42
    )

    rows = []

    for fold_id, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        X_tr = X.iloc[train_idx].copy()
        X_va = X.iloc[valid_idx].copy()

        y_tr = y.iloc[train_idx].copy()
        y_va = y.iloc[valid_idx].copy()

        model, fold_scaler, _ = train_mlp(
            X_tr,
            y_tr,
            X_va,
            y_va,
            epochs=epochs,
            batch_size=32,
            learning_rate=1e-3,
            weight_decay=1e-4,
            patience=30,
            seed=42 + fold_id
        )

        model.eval()

        with torch.no_grad():
            X_va_scaled = fold_scaler.transform(X_va)
            X_va_tensor = torch.tensor(
                X_va_scaled, dtype=torch.float32
            ).to(DEVICE)

            logits = model(X_va_tensor)
            prob = torch.sigmoid(logits).cpu().numpy()

        metrics = evaluate_predictions(y_va, prob)

        row = {
            "Fold": fold_id,
            **metrics
        }

        rows.append(row)

        if fold_id % 10 == 0:
            print(f"Completed {fold_id}/{n_splits*n_repeats} folds")

    return pd.DataFrame(rows)


mlp_cv = repeated_cv_mlp(
    X, y,
    n_splits=10,
    n_repeats=5,
    epochs=250
)

display(mlp_cv.head())


In [ ]:

# Summary of repeated CV performance

metric_columns = [
    "Accuracy",
    "Balanced_Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC",
    "PR_AUC",
    "LogLoss",
    "Brier"
]

mlp_cv_summary = pd.DataFrame({
    "Mean": mlp_cv[metric_columns].mean(),
    "SD": mlp_cv[metric_columns].std()
})

display(mlp_cv_summary)



## 9. Compare MLP against classical baselines

We add Logistic Regression and Random Forest using the same repeated CV framework.

This is not intended to reproduce every experiment from the earlier notebook; it provides a direct DL-vs-classical comparison on the same features and folds.


In [ ]:

def sklearn_repeated_cv(model, X, y, n_splits=10, n_repeats=5):
    cv = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=42
    )

    rows = []

    for fold_id, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        X_tr = X.iloc[train_idx]
        X_va = X.iloc[valid_idx]
        y_tr = y.iloc[train_idx]
        y_va = y.iloc[valid_idx]

        model.fit(X_tr, y_tr)
        prob = model.predict_proba(X_va)[:, 1]

        rows.append({
            "Fold": fold_id,
            **evaluate_predictions(y_va, prob)
        })

    return pd.DataFrame(rows)


lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

lr_cv = sklearn_repeated_cv(lr_model, X, y)
rf_cv = sklearn_repeated_cv(rf_model, X, y)

comparison = pd.DataFrame({
    "MLP": mlp_cv[metric_columns].mean(),
    "Logistic_Regression": lr_cv[metric_columns].mean(),
    "Random_Forest": rf_cv[metric_columns].mean()
})

display(comparison)



# 10. ROC and Precision-Recall curves on the 70/30 experiment

These curves are illustrative for the single split. They should not be interpreted as a definitive estimate of generalization.


In [ ]:

# MLP curves
fpr, tpr, _ = roc_curve(y_test, test_prob)
precision, recall, _ = precision_recall_curve(y_test, test_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"MLP ROC Curve (AUC = {roc_auc_score(y_test, test_prob):.3f})")
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"MLP Precision-Recall Curve (AP = {average_precision_score(y_test, test_prob):.3f})")
plt.show()



# 11. Threshold analysis

A 0.5 probability threshold is not necessarily optimal for a clinical prediction problem.

We inspect thresholds from 0.10 to 0.90.

This is an analysis of the **single held-out test split**, so threshold selection should not be presented as an independently validated clinical cutoff.


In [ ]:

threshold_rows = []

for threshold in np.arange(0.10, 0.91, 0.05):
    metrics = evaluate_predictions(
        y_test,
        test_prob,
        threshold=threshold
    )

    threshold_rows.append({
        "Threshold": round(threshold, 2),
        "Accuracy": metrics["Accuracy"],
        "Balanced_Accuracy": metrics["Balanced_Accuracy"],
        "Precision": metrics["Precision"],
        "Recall": metrics["Recall"],
        "F1": metrics["F1"]
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)



# 12. Confusion matrix at the default threshold


In [ ]:

test_pred = (test_prob >= 0.5).astype(int)

cm = confusion_matrix(y_test, test_pred)

print(cm)
print()
print(classification_report(
    y_test,
    test_pred,
    target_names=["Survived", "Died"],
    zero_division=0
))



# 13. Check for overfitting

A neural network can easily memorize a dataset this small.

We compare training and validation loss from the single split and inspect the variability of repeated CV.

A high training performance combined with unstable validation performance would be evidence against claiming that DL is genuinely superior.


In [ ]:

print("Single-split final training loss:",
      history["train_loss"][-1])

print("Single-split final validation loss:",
      history["valid_loss"][-1])

print("\nRepeated-CV MLP:")
display(mlp_cv_summary.loc[
    ["Accuracy", "Balanced_Accuracy", "F1", "ROC_AUC", "PR_AUC"]
])



# 14. Feature importance: permutation importance for the MLP

Neural networks do not provide a simple tree-style feature importance.

We therefore use permutation importance on the held-out 70/30 test set.

The feature is shuffled while all other features remain unchanged. The decrease in ROC-AUC indicates how much predictive information the model relied on from that feature.

This is **predictive importance, not causal importance**.


In [ ]:

rng = np.random.default_rng(42)

baseline_auc = roc_auc_score(y_test, test_prob)

importance_rows = []

X_test_original = X_test.copy()

for feature in FEATURES:
    drops = []

    for _ in range(30):
        X_perm = X_test_original.copy()
        X_perm[feature] = rng.permutation(X_perm[feature].values)

        X_perm_scaled = scaler.transform(X_perm)

        with torch.no_grad():
            tensor = torch.tensor(
                X_perm_scaled,
                dtype=torch.float32
            ).to(DEVICE)

            perm_prob = torch.sigmoid(
                mlp(tensor)
            ).cpu().numpy()

        perm_auc = roc_auc_score(y_test, perm_prob)
        drops.append(baseline_auc - perm_auc)

    importance_rows.append({
        "Feature": feature,
        "Mean_ROC_AUC_Drop": np.mean(drops),
        "SD": np.std(drops)
    })

mlp_importance = pd.DataFrame(
    importance_rows
).sort_values(
    "Mean_ROC_AUC_Drop",
    ascending=False
)

display(mlp_importance)


In [ ]:

plt.figure(figsize=(9, 6))

plot_df = mlp_importance.sort_values(
    "Mean_ROC_AUC_Drop"
)

plt.barh(
    plot_df["Feature"],
    plot_df["Mean_ROC_AUC_Drop"]
)

plt.xlabel("Decrease in ROC-AUC after permutation")
plt.ylabel("Feature")
plt.title("MLP Held-out Permutation Importance")
plt.show()



# 15. Optional: learning-rate sensitivity

Because the dataset is small, a single learning rate can produce an unstable result.

We compare a few conservative learning rates on the same 70/30 split.

This is a sensitivity analysis, not test-set hyperparameter tuning for a final reported model.


In [ ]:

learning_rates = [1e-4, 3e-4, 1e-3, 3e-3]

lr_results = []

for lr in learning_rates:
    model_lr, scaler_lr, hist_lr = train_mlp(
        X_train,
        y_train,
        X_test,
        y_test,
        epochs=300,
        batch_size=32,
        learning_rate=lr,
        weight_decay=1e-4,
        patience=40,
        seed=42
    )

    model_lr.eval()

    with torch.no_grad():
        Xt = scaler_lr.transform(X_test)
        Xt = torch.tensor(Xt, dtype=torch.float32).to(DEVICE)
        p = torch.sigmoid(model_lr(Xt)).cpu().numpy()

    result = evaluate_predictions(y_test, p)

    lr_results.append({
        "Learning_Rate": lr,
        **result
    })

learning_rate_df = pd.DataFrame(lr_results)
display(learning_rate_df)



# 16. Final DL conclusion

Use the following logic when interpreting the result:

### If MLP ROC-AUC is clearly higher and stable

A result such as:

- MLP repeated-CV ROC-AUC consistently above RF/XGB
- relatively small standard deviation
- no severe train/validation gap

would support the claim that the neural network adds predictive value.

### If MLP is similar to RF

That is also a useful result.

For a 299-patient tabular dataset, a simpler model performing similarly to a neural network can be preferable because it is easier to interpret and less computationally complex.

### If MLP is worse

That is not a failure of the analysis.

It would indicate that deep learning does not provide a clear advantage for this small tabular dataset.

### Most important

Do **not** select a model because it produces an accuracy close to the paper's 97.78%.

The scientific question is whether the result is:

1. reproducible,
2. leakage-free,
3. stable across resampling,
4. properly calibrated,
5. and genuinely better than strong classical baselines.


In [ ]:

# 17. Export results

output_file = "heart_failure_deep_learning_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    pd.DataFrame([single_result], index=["MLP_70_30"]).to_excel(
        writer, sheet_name="mlp_holdout"
    )
    mlp_cv.to_excel(writer, sheet_name="mlp_repeated_cv", index=False)
    mlp_cv_summary.to_excel(writer, sheet_name="mlp_cv_summary")
    comparison.to_excel(writer, sheet_name="model_comparison")
    threshold_df.to_excel(writer, sheet_name="threshold_analysis", index=False)
    mlp_importance.to_excel(writer, sheet_name="mlp_importance", index=False)
    learning_rate_df.to_excel(writer, sheet_name="learning_rate", index=False)

print(f"Saved: {output_file}")
